In [3]:
!nvidia-smi

Sat Aug 15 20:21:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
!pip install nvcc4jupyter
%load_ext nvcc4jupyter

Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmp3md_a5c6".


In [5]:
%%cuda
#include <iostream>
#include <cuda_runtime.h>
#include <stdio.h>

// ==========================================
// CHALLENGE 1: VECTOR ADDITION (PHASE 2A)
// ==========================================

// TODO: Write your __global__ vectorAdd kernel
// 1. Calculate thread global index 'i' using blockIdx.x, blockDim.x, and threadIdx.x
// 2. Add bounds check (i < numElements)
// 3. Perform addition: C[i] = A[i] + B[i]

__global__
void vectorAdd(int* A, int* B, int* C, int n) {
    int i = threadIdx.x + blockDim.x * blockIdx.x;
    if (i < n) {
        C[i] = A[i] + B[i];
    }
}

int main() {
    // TODO 1: Define host data (two 1D arrays A and B with at least 10 elements)
    int A_h[100] = {
    84, 12, 95, 37, 48, 61, 23, 79, 10, 52,
    91, 14, 63, 88, 29, 45, 76, 33, 58, 97,
    19, 82, 41, 67, 25, 93, 11, 54, 70, 36,
    89, 22, 47, 65, 31, 80, 15, 59, 94, 28,
    73, 17, 62, 38, 85, 21, 50, 96, 42, 69,
    13, 78, 24, 87, 35, 60, 18, 92, 49, 71,
    30, 83, 16, 55, 98, 27, 64, 40, 77, 20,
    90, 34, 57, 81, 26, 68, 43, 99, 15, 72,
    32, 86, 51, 66, 39, 95, 12, 74, 23, 56,
    88, 44, 70, 19, 82, 31, 63, 46, 91, 25
};
    int B_h[100] = {
    16, 88,  5, 63, 52, 39, 77, 21, 90, 48,
     9, 86, 37, 12, 71, 55, 24, 67, 42,  3,
    81, 18, 59, 33, 75,  7, 89, 46, 30, 64,
    11, 78, 53, 35, 69, 20, 85, 41,  6, 72,
    27, 83, 38, 62, 15, 79, 50,  4, 58, 31,
    87, 22, 76, 13, 65, 40, 82,  8, 51, 29,
    70, 17, 84, 45,  2, 73, 36, 60, 23, 80,
    10, 66, 43, 19, 74, 32, 57,  1, 85, 28,
    68, 14, 49, 34, 61,  5, 88, 26, 77, 44,
    12, 56, 30, 81, 18, 69, 37, 54,  9, 75
};
    int C_h[100] = {};
    // TODO 2: Calculate size in bytes
    size_t array_size = sizeof(A_h);

    // TODO 3: Declare device pointers (d_A, d_B, d_C) and allocate GPU memory using cudaMalloc
    int *A_d, *B_d, *C_d;

    cudaMalloc((void **) &A_d, array_size);
    cudaMalloc((void **) &B_d, array_size);
    cudaMalloc((void **) &C_d, array_size);

    // TODO 4: Copy host arrays to device using cudaMemcpy (cudaMemcpyHostToDevice)
    cudaMemcpy(A_d, A_h, array_size, cudaMemcpyHostToDevice);
    cudaMemcpy(B_d, B_h, array_size, cudaMemcpyHostToDevice);

    // TODO 5: Define block/grid dimensions and launch your kernel: kernelName<<<blocks, threads>>>(...)
    int threads = 256;
    int blocks = ceil(100 / 256.0);

    vectorAdd<<<blocks, threads>>>(A_d, B_d, C_d, 100);

    // TODO 6: Call cudaDeviceSynchronize() and copy result C back to CPU (cudaMemcpyDeviceToHost)
    cudaDeviceSynchronize();
    cudaMemcpy(C_h, C_d, array_size, cudaMemcpyDeviceToHost);
    // TODO 7: Print host result to verify math correctness
    for (int i = 0; i < 10; i++) {
        printf("%d + %d = %d\n", A_h[i], B_h[i], C_h[i]);
    }


    // TODO 8: Free device memory using cudaFree
    cudaFree(A_d);
    cudaFree(B_d);
    cudaFree(C_d);
    return 0;
}

84 + 16 = 100
12 + 88 = 100
95 + 5 = 100
37 + 63 = 100
48 + 52 = 100
61 + 39 = 100
23 + 77 = 100
79 + 21 = 100
10 + 90 = 100
52 + 48 = 100



In [19]:
%%cuda
#include <iostream>
#include <cuda_runtime.h>


__global__
void matrixMult(int *A, int *B, int *C, int inner_dim, int w, int h)
{
    int col = blockDim.x * blockIdx.x + threadIdx.x;
    int row = blockDim.y * blockIdx.y + threadIdx.y;

    // we need something to know the future m*n
    // we need to know the inner dimesion of the matrix a
    if (col < w && row < h)
    {
        int sum = 0;
        for (int i = 0; i < inner_dim; i++) {
            sum += A[row*w +i] * B[i*w + col];
        }

        C[row * w + col] = sum;
    }
}

// main function for running on the host
int main()
{
    // create the main host memory variables
    int A_h[100] = {
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, // Row 0
        2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, // Row 1
        3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, // Row 2
        4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, // Row 3
        5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5  // Row 4
    };

    int B_h[100] = {
        1, 2, 3, 4, 5, // Row 0
        1, 2, 3, 4, 5, // Row 1
        1, 2, 3, 4, 5, // Row 2
        1, 2, 3, 4, 5, // Row 3
        1, 2, 3, 4, 5, // Row 4
        1, 2, 3, 4, 5, // Row 5
        1, 2, 3, 4, 5, // Row 6
        1, 2, 3, 4, 5, // Row 7
        1, 2, 3, 4, 5, // Row 8
        1, 2, 3, 4, 5, // Row 9
        1, 2, 3, 4, 5, // Row 10
        1, 2, 3, 4, 5, // Row 11
        1, 2, 3, 4, 5, // Row 12
        1, 2, 3, 4, 5, // Row 13
        1, 2, 3, 4, 5, // Row 14
        1, 2, 3, 4, 5, // Row 15
        1, 2, 3, 4, 5, // Row 16
        1, 2, 3, 4, 5, // Row 17
        1, 2, 3, 4, 5, // Row 18
        1, 2, 3, 4, 5  // Row 19
    };
    int C_h[100];

    // create the main device memory variables
    int *A_d, *B_d, *C_d;

    size_t array_size = sizeof(A_h);

    // allocate the device memory vairable
    cudaMalloc((void **)&A_d, array_size);
    cudaMalloc((void **)&B_d, array_size);
    cudaMalloc((void **)&C_d, array_size);
    // copy the memory from the host variabels to the device variables

    cudaMemcpy(A_d, A_h, array_size, cudaMemcpyHostToDevice);
    cudaMemcpy(B_d, B_h, array_size, cudaMemcpyHostToDevice);

    // call the kernel
    dim3 threads(5,5);
    dim3 blocks(1,1);

    matrixMult<<<blocks, threads>>>(A_d, B_d, C_d, 20, 5, 5);

    // transfer the memory from device to host
    cudaMemcpy(C_h, C_d, array_size, cudaMemcpyDeviceToHost);

    // print the new host variable

    for (int i = 0; i < 10; i++)
    {
        std::cout << C_h[i] << std::endl;
    }

    // clear the memory
    cudaFree(&A_d);
    cudaFree(&B_d);
    cudaFree(&C_d);

    return 0;
}

20
40
60
80
100
25
50
75
100
125

